In [42]:
!wget -O bunny.off https://raw.githubusercontent.com/pmp-library/pmp-library/main/data/off/bunny.off

--2026-03-20 15:06:10--  https://raw.githubusercontent.com/pmp-library/pmp-library/main/data/off/bunny.off
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2724384 (2.6M) [text/plain]
Saving to: ‘bunny.off’

bunny.off           100%[===================>]   2.60M  7.43MB/s    in 0.3s    

2026-03-20 15:06:11 (7.43 MB/s) - ‘bunny.off’ saved [2724384/2724384]



In [65]:
%matplotlib qt
import igl
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse.linalg as sla
import plotly.graph_objects as go
import scipy.sparse as sp
from pathlib import Path
from dataclasses import dataclass
from plotly.subplots import make_subplots
from torch_geometric.datasets import TOSCA

ModuleNotFoundError: No module named 'torch'

## Some useful funcitons

In [44]:
def plotly_mesh(V, F, color='lightblue', opacity=0.8, flatshading=True, title="Mon Maillage 3D"):
    """
    Affiche un maillage 3D à partir des sommets V et des faces F avec Plotly.

    Paramètres :
    - V : numpy array (N, 3) contenant les coordonnées [x, y, z] des sommets.
    - F : numpy array (M, 3) contenant les indices [i, j, k] des faces.
    - color : Couleur du maillage (str ou code hexadécimal).
    - opacity : Niveau de transparence entre 0 et 1.
    - flatshading : True pour des facettes marquées, False pour un lissage.
    - title : Titre du graphique.
    """

    # Sécurité : on s'assure que ce sont bien des tableaux NumPy
    V = np.asarray(V)
    F = np.asarray(F)

    # Extraction des coordonnées et des indices
    x, y, z = V[:, 0], V[:, 1], V[:, 2]
    i, j, k = F[:, 0], F[:, 1], F[:, 2]

    # Création du mesh Plotly
    mesh = go.Mesh3d(
        x=x, y=y, z=z,
        i=i, j=j, k=k,
        color=color,
        opacity=opacity,
        flatshading=flatshading
    )

    # Initialisation de la figure
    fig = go.Figure(data=[mesh])

    # Mise en forme de la scène
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title='Axe X',
            yaxis_title='Axe Y',
            zaxis_title='Axe Z'
        ),
        margin=dict(l=0, r=0, b=0, t=40) # Réduit les marges autour du graphique 3D
    )


    # On retourne la figure au cas où vous voudriez la modifier ou la sauvegarder plus tard
    return fig

In [45]:
def plotly_complete_frames(V, F, U, n_samples=200):
    """
    Visualise le repère complet (u_M, u_m, n) sur un échantillon de faces à l'aide de Plotly.
    """
    num_faces = F.shape[0]
    face_centers = np.mean(V[F], axis=1)

    # Échantillonnage aléatoire
    idx = np.random.choice(np.arange(num_faces), size=int(n_samples), replace=False)

    centers = face_centers[idx]
    u_M = U[idx, :, 0]
    u_m = U[idx, :, 1]
    n = U[idx, :, 2]

    # Création du maillage de base
    title = f"Local Frames ({len(idx)} samples)<br>Red: u_M, Green: u_m, Blue: n"
    fig = plot_mesh(V, F, c="lightgrey", title=title)

    # Calcul de la longueur des vecteurs (proportionnelle à la taille de l'objet)
    length = np.linalg.norm(V.max(0) - V.min(0)) * 0.04

    # --- Fonction interne pour dessiner les vecteurs ---
    def add_quiver_plotly(fig, origins, vectors, color, name):
        endpoints = origins + vectors * length

        # Astuce Plotly : on entrelace [origine, extremite, None]
        # pour dessiner des milliers de lignes indépendantes en un seul tracé (très rapide)
        x_lines, y_lines, z_lines = [], [], []
        for o, e in zip(origins, endpoints):
            x_lines.extend([o[0], e[0], None])
            y_lines.extend([o[1], e[1], None])
            z_lines.extend([o[2], e[2], None])

        fig.add_trace(go.Scatter3d(
            x=x_lines, y=y_lines, z=z_lines,
            mode='lines',
            line=dict(color=color, width=4),
            name=name,
            hoverinfo='skip' # Évite que l'infobulle gêne la lecture
        ))

    # Ajout des 3 directions
    add_quiver_plotly(fig, centers, u_M, 'red', 'u_M (Max)')
    add_quiver_plotly(fig, centers, u_m, 'green', 'u_m (Min)')
    add_quiver_plotly(fig, centers, n, 'blue', 'Normale')

    return fig

In [46]:
def generer_selle_igl(nombre_sommets_cible):
    """
    Génère un maillage de selle de cheval (paraboloïde hyperbolique) pour libigl.

    Retourne :
    V : numpy array (N, 3) des coordonnées des sommets.
    F : numpy array (M, 3) des indices des faces (triangles).
    indice_centre : entier, l'indice du sommet situé exactement au point col.
    """
    # 1. Calculer la taille de la grille (n x n)
    # On force 'n' à être un nombre impair pour avoir un sommet central exact.
    n = int(np.round(np.sqrt(nombre_sommets_cible)))
    if n % 2 == 0:
        n += 1

    print(f"Génération d'une grille {n}x{n} (Total: {n*n} sommets)")

    # 2. Créer les coordonnées X et Y entre -1 et 1
    x = np.linspace(-1, 1, n)
    y = np.linspace(-1, 1, n)
    X, Y = np.meshgrid(x, y)

    # 3. Appliquer l'équation de la selle : Z = X^2 - Y^2
    Z = X**2 - Y**2

    # 4. Construire la matrice des sommets V pour libigl (N, 3)
    V = np.column_stack((X.flatten(), Y.flatten(), Z.flatten()))

    # 5. Construire la matrice des faces F (triangles) (M, 3)
    F = []
    for i in range(n - 1):
        for j in range(n - 1):
            # Indices des 4 coins de la cellule courante
            v0 = i * n + j
            v1 = v0 + 1
            v2 = (i + 1) * n + j
            v3 = v2 + 1

            # Diviser le quadrilatère en 2 triangles pour libigl
            F.append([v0, v1, v2])
            F.append([v1, v3, v2])

    F = np.array(F, dtype=np.int32)

    # 6. Trouver l'indice du point central (point selle)
    centre_i = n // 2
    centre_j = n // 2
    indice_centre = centre_i * n + centre_j

    return V, F, indice_centre

In [47]:
def plot_mesh(X, F, subplot=[1, 1, 1], title="", el=0, az=0, lwdt=.1, dist=6, c="grey",
              face_func=None, vertex_func=None):
    X = X.T
    ax = plt.subplot(subplot[0], subplot[1], subplot[2], projection='3d')
    collec = ax.plot_trisurf(X[0, :], X[1, :], X[2, :], triangles=F, lw=lwdt, color=c, alpha=1)
    if face_func is not None:
        collec.set_array(face_func)
    if vertex_func is not None:
        vertex_value = lambda v: np.mean(np.stack([v[F[:, 0]], v[F[:, 1]], v[F[:, 2]]]), axis=0)
        collec.set_array(vertex_value(vertex_func))
    ax.axis("off")
    # ax.set_aspect('equal')
    ax.view_init(elev=el, azim=az)
    ax.dist = dist

    # Create cubic bounding box to simulate equal aspect ratio
    # this is a hack to prevent a change in aspect ratio during rotation
    x = X[0, :]
    y = X[1, :]
    z = X[2, :]
    max_range = np.array([x.max() - x.min(), y.max() - y.min(), z.max() - z.min()]).max()
    Xb = 0.5 * max_range * np.mgrid[-1:2:2, -1:2:2, -1:2:2][0].flatten() + 0.5 * (x.max() + x.min())
    Yb = 0.5 * max_range * np.mgrid[-1:2:2, -1:2:2, -1:2:2][1].flatten() + 0.5 * (y.max() + y.min())
    Zb = 0.5 * max_range * np.mgrid[-1:2:2, -1:2:2, -1:2:2][2].flatten() + 0.5 * (z.max() + z.min())
    for xb, yb, zb in zip(Xb, Yb, Zb):
        ax.plot([xb], [yb], [zb], 'w')

    plt.title(title)
    return ax  # need the ax object for the last section of the lab

In [48]:
def plot_complete_frames(V, F, U, n_samples=200, el=0, az=0):
    """
    Visualise le repère complet (u_M en rouge, u_m en vert, n en bleu).
    n_samples : Nombre total de repères à afficher.
    seed : Graine pour figer le tirage aléatoire des faces.
    el : Angle d'élévation (caméra).
    az : Angle d'azimut (caméra).
    """
    # 1. Fixation de la graine aléatoire si elle est fournie

    num_faces = F.shape[0]
    face_centers = np.mean(V[F], axis=1)

    idx = np.arange(num_faces)
    idx = np.random.choice(idx, size=int(n_samples), replace=False)

    centers = face_centers[idx]
    u_M = U[idx, :, 0]
    u_m = U[idx, :, 1]
    n = U[idx, :, 2]

    # 2. Affichage du maillage en passant les paramètres de caméra (el et az)
    ax = plot_mesh(V, F, c="lightgrey", lwdt=0.05,
                   title=f"Local Frames ({len(idx)} samples)\nRed: u_M, Green: u_m, Blue: n",
                   el=el, az=az)

    # Taille des vecteurs proportionnelle à la boîte englobante de l'objet
    length = np.linalg.norm(V.max(0) - V.min(0)) * 0.04

    # Direction de courbure MAX (u_M) - Rouge
    ax.quiver(centers[:,0], centers[:,1], centers[:,2],
              u_M[:,0], u_M[:,1], u_M[:,2], color='red', length=length)

    # Direction de courbure MIN (u_m) - Vert
    ax.quiver(centers[:,0], centers[:,1], centers[:,2],
              u_m[:,0], u_m[:,1], u_m[:,2], color='green', length=length)

    # Normale (n) - Bleu
    ax.quiver(centers[:,0], centers[:,1], centers[:,2],
              n[:,0], n[:,1], n[:,2], color='blue', length=length)

    return ax

In [49]:
def plot_diffusion_tensor(V, F, D_F, n_samples=200, el=0, az=0):

    m = F.shape[0]

    eigvals, eigvecs = np.linalg.eigh(D_F)

    principal_dirs = eigvecs[:, :, 2]

    idx = np.random.choice(np.arange(m), size=int(n_samples), replace=False)
    face_centers = np.mean(V[F], axis=1)

    centers = face_centers[idx]
    dirs = principal_dirs[idx]

    # Création du plot avec votre fonction plot_mesh
    title = f"Directions Principales de Diffusion ({n_samples} faces)\nRouge = Étirement maximal"
    ax = plot_mesh(V, F, c="lightgrey", lwdt=0.05, title=title, el=el, az=az)

    # Longueur des flèches proportionnelle à la taille de l'objet
    length = np.linalg.norm(V.max(0) - V.min(0)) * 0.04

    # Affichage des vecteurs principaux en rouge
    ax.quiver(centers[:, 0], centers[:, 1], centers[:, 2],
              dirs[:, 0], dirs[:, 1], dirs[:, 2],
              color='red', length=length, normalize=True)

    return ax

## 0. Definitions

In [50]:
@dataclass
class Options:
    alpha: float
    angle: float
    tau: float

In [51]:
def plot_mesh(V, F, face_func=None, vertex_func=None):
    V = np.asarray(V)
    F = np.asarray(F)

    # 1. Configuration des paramètres de base pour les faces
    mesh_kwargs = {
        'x': V[:, 0], 'y': V[:, 1], 'z': V[:, 2],
        'i': F[:, 0], 'j': F[:, 1], 'k': F[:, 2],
        'opacity': 1.0, # Mieux vaut 1.0 quand on utilise une carte de couleurs
        'flatshading': True,
        'name': 'Faces'
    }

    # Gestion des couleurs selon face_func ou vertex_func
    if face_func is not None:
        mesh_kwargs['intensity'] = face_func
        mesh_kwargs['intensitymode'] = 'cell'
        mesh_kwargs['colorscale'] = 'Viridis' # Vous pouvez changer la palette ici
    elif vertex_func is not None:
        mesh_kwargs['intensity'] = vertex_func
        mesh_kwargs['intensitymode'] = 'vertex'
        mesh_kwargs['colorscale'] = 'Viridis'
    else:
        # Couleur par défaut si aucune fonction n'est passée
        mesh_kwargs['color'] = 'lightblue'
        mesh_kwargs['opacity'] = 0.8

    mesh_trace = go.Mesh3d(**mesh_kwargs)

    # 2. Extraction des arêtes uniques
    edges = np.vstack([F[:, [0, 1]], F[:, [1, 2]], F[:, [2, 0]]])
    unique_edges = np.unique(np.sort(edges, axis=1), axis=0)

    # 3. Préparation des coordonnées des arêtes
    x_edges = np.empty(3 * len(unique_edges))
    y_edges = np.empty(3 * len(unique_edges))
    z_edges = np.empty(3 * len(unique_edges))

    x_edges[0::3] = V[unique_edges[:, 0], 0]
    x_edges[1::3] = V[unique_edges[:, 1], 0]
    x_edges[2::3] = np.nan

    y_edges[0::3] = V[unique_edges[:, 0], 1]
    y_edges[1::3] = V[unique_edges[:, 1], 1]
    y_edges[2::3] = np.nan

    z_edges[0::3] = V[unique_edges[:, 0], 2]
    z_edges[1::3] = V[unique_edges[:, 1], 2]
    z_edges[2::3] = np.nan

    # 4. Création de la trace pour les arêtes
    edge_trace = go.Scatter3d(
        x=x_edges, y=y_edges, z=z_edges,
        mode='lines',
        line=dict(color='black', width=2),
        name='Arêtes',
        hoverinfo='skip'
    )

    # 5. Assemblage et affichage
    fig = go.Figure(data=[mesh_trace, edge_trace])
    fig.update_layout(
        scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
        margin=dict(l=0, r=0, b=0, t=30)
    )

    return fig

## 1. Load mesh

In [69]:
mesh_path = Path('./data/TOSCA/raw/Meshes/cat0.off')
V, F = igl.read_triangle_mesh(mesh_path)

In [67]:
from torch_geometric.datasets import TOSCA

# 1. Charger le dataset pour la catégorie 'Cat' (Chat)
# PyG va télécharger automatiquement les fichiers dans le dossier 'data/TOSCA' s'ils n'y sont pas.
dataset = TOSCA(root='data/TOSCA', categories=['Cat'])

# 2. Sélectionner un chat spécifique (par exemple, le premier chat, index 0)
data = dataset[0]

# 3. Extraire V et F en tableaux NumPy
V = data.pos.numpy()
F = data.face.t().numpy()

URLError: <urlopen error [Errno -2] Name or service not known>

In [70]:
V, F, J, I = igl.decimate(V, F, 5000)

In [53]:
#V, F, centre = generer_selle_igl(500)

In [71]:
fig = plot_mesh(V,F)
fig.show()

## 2. Extracting Local Frames and Principal Curvatures

To construct the Finsler-Laplace-Beltrami Operator (FLBO), we first need to define an anisotropic metric on the surface. Unlike standard isotropic diffusion, the FLBO diffuses heat differently along specific geometric directions. To achieve this, we must build a local coordinate system for every triangle in the mesh.

Following the paper's discretization scheme, for each triangle $ijk \in F$, we define an orthonormal reference frame $U_{ijk}$.

$$U_{ijk} = (\hat{u}_{M}, \hat{u}_{m}, \hat{n})$$

Where:
* $\hat{n} \in \mathbb{R}^3$ is the unit normal vector of the face.
* $\hat{u}_{M} \in \mathbb{R}^3$ is the direction of maximum principal curvature.
* $\hat{u}_{m} \in \mathbb{R}^3$ is the direction of minimum principal curvature.

**Implementation note:** While $\hat{n}$ is computed per face, standard algorithms (like those in `libigl`) compute principal curvatures at the vertices. To obtain a valid orthonormal frame $U_{ijk}$ per face, we take the maximum principal curvature direction from one of the face's vertices, project it onto the face's tangent plane (to guarantee orthogonality with $\hat{n}$), and then compute $\hat{u}_{m}$ using the cross product $\hat{n} \times \hat{u}_{M}$.

In [55]:
def compute_local_frames(V, F):
    """
    Calcule le repère orthonormé U_ijk = (u_M, u_m, n) pour chaque face.
    Retourne un tenseur U de forme (nombre_de_faces, 3, 3).
    """
    n = igl.per_face_normals(V, F, np.array([0.0, 0.0, 1.0]))
    
    pd1, pd2, pv1, pv2, bad_vertices = igl.principal_curvature(V, F)

    num_faces = F.shape[0]

    U = np.zeros((num_faces, 3, 3))

    for f_idx in range(num_faces):
        face = F[f_idx]
        n_f = n[f_idx]

        dir_max = pd1[face[0]]

        # Projection de dir_max sur le plan tangent de la face
        u_M_f = dir_max - np.dot(dir_max, n_f) * n_f

        norm = np.linalg.norm(u_M_f)
        if norm > 1e-8:
            u_M_f = u_M_f / norm
        else:
            edge = V[face[1]] - V[face[0]]
            u_M_f = edge - np.dot(edge, n_f) * n_f
            u_M_f = u_M_f / np.linalg.norm(u_M_f)

        u_m_f = np.cross(n_f, u_M_f)

        U[f_idx, :, :] = [u_M_f, u_m_f, n_f]

    return U

## 3. Computing the Finsler Diffusivity Tensor ($D_{\mathcal{F}^*}$)

Le tenseur de diffusion de Finsler, noté $D_{Finsler}$, permet de généraliser l'opérateur de Laplace-Beltrami standard pour modéliser une diffusion **anisotrope** (qui privilégie certaines directions) et **asymétrique** (où aller de A à B n'a pas le même coût que de B à A) sur une surface.

Cette asymétrie est particulièrement utile pour l'analyse de formes et les correspondances (comme dans l'algorithme FLBO), car elle agit comme un champ de vent directionnel sur la variété.

Voici les étapes clés de sa construction :

**1. Repère local et Rotation**
Pour chaque face du maillage, on dispose d'un repère orthonormé local $(u_M, u_m, n)$ aligné avec les directions de courbure principale. On applique une rotation d'un angle donné dans le plan tangent pour obtenir un nouveau repère $(U_1, U_2, n)$ :
$$U_1 = \cos(\theta)u_m + \sin(\theta)u_M$$
$$U_2 = \cos(\theta)u_M - \sin(\theta)u_m$$

**2. Matrice de Diffusion Riemannienne de base ($M$)**
On construit d'abord une métrique Riemannienne standard $M$ qui pondère la diffusion selon les courbures (via le paramètre $\alpha$) :
$$M = D_1 (U_1 U_1^T) + D_2 (U_2 U_2^T) + D_3 (n n^T)$$
Où $D_1 = \frac{1}{1+\alpha}$, et $D_2 = D_3 = 1$.



**3. Métrique de Randers et Vecteur de Dérive ($w$)**
Pour introduire l'asymétrie, la métrique de Finsler utilisée ici est une **métrique de Randers**. Elle est définie par la donnée de la métrique Riemannienne $M$ et d'un champ de vecteurs $w$.
Ici, la dérive est orientée le long de la direction de courbure maximale tournée, modulée par un coefficient $\tau$ :
$$w = \tau U_2$$

**4. Espace Dual et Tenseurs Conjugués**
Pour intégrer cette métrique dans l'opérateur de Laplace (qui travaille sur le gradient, un covecteur), on doit passer dans l'espace dual. On calcule d'abord le facteur d'échelle $\eta$ lié à la norme Riemannienne de $w$ :
$$\eta = 1 - \langle w, M w \rangle$$

Puis on dérive le tenseur dual $M^*$ et le vecteur dual $w^*$ :
$$M^* = \frac{(Mw)(Mw)^T + \eta M}{\eta^2}$$
$$w^* = -\frac{Mw}{\eta}$$

**5. Tenseur de Finsler Final**
Le tenseur final, qui sera utilisé dans la forme bilinéaire de la matrice de rigidité, combine ces éléments duaux :
$$D_{Finsler} = M^* - w^* (w^*)^T$$

> **Note d'optimisation :** Mathématiquement, en développant $w^* (w^*)^T$, le terme lié au produit externe de $Mw$ s'annule avec celui de $M^*$. L'expression complète se simplifie théoriquement en $D_{Finsler} = \frac{M}{\eta}$. Le code implémente la formulation complète pour rester fidèle à la définition générale de la métrique de Randers.

In [56]:
def compute_D_finsler(U: np.ndarray, options: Options) -> np.ndarray:
    """
    Calcule le tenseur de métrique de Finsler (D_finsler) de façon hyper-optimisée.
    """
    # 1. Matrice de rotation dans le plan tangent
    c = np.cos(options.angle)
    s = np.sin(options.angle)
    R = np.array([
        [ s,  c, 0],  # U1
        [ c, -s, 0],  # U2
        [ 0,  0, 1]   # N
    ])

    # Application de la rotation à tous les repères d'un coup
    U_rot = R @ U  # (3, 3) @ (N_faces, 3, 3) -> U_rot a (U1, U2, N) en lignes

    # 2. Construction de la matrice de diffusion de base M = U^T * D * U
    D_mat = np.diag([1.0 / (1.0 + options.alpha), 1.0, 1.0])

    # U_rot.transpose(0, 2, 1) est U^T pour chaque face
    M = U_rot.transpose(0, 2, 1) @ D_mat @ U_rot

    # 3. Métrique de Randers et dualité
    # On sait mathématiquement que <w, Mw> = tau^2 puisque w = tau * U2 et D2 = 1
    eta = 1.0 - (options.tau ** 2)

    # Extraction de U2 (qui est sur la 2ème ligne de U_rot, index 1)
    U2 = U_rot[:, 1, :]
    w = options.tau * U2

    # Calcul de w* (vecteur dual)
    # np.einsum('mij,mj->mi', M, w) est le produit matrice-vecteur M @ w
    Mw = np.einsum('mij,mj->mi', M, w)
    wstar = -Mw / eta

    # Produit externe w* @ w*^T
    wstar_outer = np.einsum('mi,mj->mij', wstar, wstar)

    # 4. Tenseur de Finsler
    # M_star = (Mw @ Mw^T + eta * M) / eta^2
    Mw_outer = np.einsum('mi,mj->mij', Mw, Mw)
    Mstar = (Mw_outer + eta * M) / (eta ** 2)

    D_finsler = Mstar - wstar_outer

    # Nettoyage des petites erreurs numériques
    D_finsler[np.abs(D_finsler) < 1e-10] = 0.0
    D_finsler[np.abs(D_finsler - 1.0) < 1e-10] = 1.0

    return D_finsler

## 4. Assembling the Anisotropic Stiffness Matrix ($W_{FLBO}$)

La matrice de rigidité, notée $W$, est la représentation discrète de l'opérateur différentiel (ici, le Laplacien de Finsler) sur le maillage triangulaire. Elle est construite en utilisant la **Méthode des Éléments Finis (FEM)**.

Plutôt que d'évaluer la diffusion sur toute la surface d'un coup, on calcule l'énergie de diffusion localement sur chaque triangle, puis on somme (on "assemble") ces contributions pour obtenir le système global.



**1. Forme bilinéaire locale (sur un triangle)**
Pour un triangle donné défini par ses trois sommets $(i_1, i_2, i_3)$, on note $e_1, e_2, e_3$ les vecteurs arêtes opposés à chaque sommet, et $\theta_{i_1}, \theta_{i_2}, \theta_{i_3}$ les angles intérieurs correspondants.

Dans un Laplacien standard, les poids reliant deux sommets dépendent uniquement de la géométrie du triangle (les fameux "poids cotangents"). Avec la métrique de Finsler, l'énergie de diffusion est déformée par le tenseur $D_{Finsler}$.

Les poids locaux calculés dans le code sont :

* **Terme extra-diagonal (interaction entre deux sommets $i_1$ et $i_2$) :**
    $$W_{i_1,i_2} = -\frac{1}{2\sin(\theta_{i_3})} e_1^T D_{Finsler} e_2$$
    Ce terme mesure à quel point le gradient de la fonction de base au sommet $i_1$ "s'aligne" avec celui du sommet $i_2$, au travers du prisme asymétrique de $D_{Finsler}$.

* **Terme diagonal (contribution du sommet $i_1$ sur lui-même) :**
    $$W_{i_1,i_1} = -\frac{\cot(\theta_{i_2}) + \cot(\theta_{i_3})}{2} e_1^T D_{Finsler} e_1$$
    Il représente l'énergie d'auto-diffusion au niveau du sommet $i_1$ à l'intérieur de ce triangle précis.

**2. Asymétrie de l'Opérateur**
Contrairement à un Laplacien classique où la matrice est symétrique ($W_{i_1,i_2} = W_{i_2,i_1}$), le tenseur $D_{Finsler}$ introduit une direction privilégiée (un "vent"). Par conséquent, l'effort pour diffuser du sommet $i_1$ vers $i_2$ n'est plus le même que de $i_2$ vers $i_1$. La matrice $W$ devient donc **non-symétrique**, ce qui est la signature mathématique de l'algorithme FLBO.

**3. Assemblage Global (Format COO)**
Une fois ces calculs effectués pour chaque triangle, la matrice globale est assemblée.
Puisqu'un sommet $i_1$ appartient à plusieurs triangles (son "1-ring"), sa valeur finale dans la matrice globale $W$ est la somme des contributions de tous les triangles adjacents.

C'est là qu'intervient le format creux **COO** (Coordinate format) de `scipy.sparse` :
1. On crée trois longues listes : `Lignes`, `Colonnes`, et `Valeurs`.
2. On y empile tous les $W_{i_1,i_2}$ et $W_{i_1,i_1}$ de chaque face.
3. Lors de l'instanciation de la matrice, `scipy` repère automatiquement les paires de coordonnées `(Ligne, Colonne)` identiques et **additionne** leurs `Valeurs`. C'est une méthode extrêmement rapide et vectorisée pour fusionner les géométries locales en une seule structure globale.

In [57]:
def compute_stiffness_matrix(V, F, D_F):
    """
    Assemble la matrice de rigidité creuse W selon la logique du code MATLAB original.
    Fidèle aux équations du papier FLBO.
    """
    n_vertices = V.shape[0]
    m_faces = F.shape[0]

    # --- 1. Calcul des angles de chaque triangle ---
    angles = np.zeros_like(F, dtype=float)
    for i in range(3):
        i1, i2, i3 = i, (i + 1) % 3, (i + 2) % 3

        pp = V[F[:, i2]] - V[F[:, i1]]
        qq = V[F[:, i3]] - V[F[:, i1]]

        # Normalisation sécurisée
        norm_pp = np.maximum(np.linalg.norm(pp, axis=1, keepdims=True), 1e-12)
        norm_qq = np.maximum(np.linalg.norm(qq, axis=1, keepdims=True), 1e-12)
        pp /= norm_pp
        qq /= norm_qq

        # Produit scalaire et angle
        dot_prod = np.sum(pp * qq, axis=1)
        dot_prod = np.clip(dot_prod, -1.0, 1.0) # Évite les NaN dus aux imprécisions flottantes
        angles[:, i1] = np.arccos(dot_prod)

    # --- 2. Construction vectorisée de la Matrice de Rigidité (W) ---
    I_W, J_W, V_W = [], [], []

    for i in range(3):
        i1, i2, i3 = i, (i + 1) % 3, (i + 2) % 3

        # Vecteurs arêtes opposés aux sommets i1 et i2
        e1 = V[F[:, i3]] - V[F[:, i2]]
        e2 = V[F[:, i1]] - V[F[:, i3]]

        norm_e1 = np.maximum(np.linalg.norm(e1, axis=1, keepdims=True), 1e-12)
        norm_e2 = np.maximum(np.linalg.norm(e2, axis=1, keepdims=True), 1e-12)
        e1 /= norm_e1
        e2 /= norm_e2

        # Calcul vectoriel des formes bilinéaires: e1^T * D_finsler * e2 et e1^T * D_finsler * e1
        # np.einsum('mij,mj->mi') fait le produit matrice-vecteur pour chaque face
        D_e2 = np.einsum('mij,mj->mi', D_F, e2)
        D_e1 = np.einsum('mij,mj->mi', D_F, e1)

        e1_D_e2 = np.sum(e1 * D_e2, axis=1)
        e1_D_e1 = np.sum(e1 * D_e1, axis=1)

        # Termes trigonométriques
        sin_i3 = np.sin(angles[:, i3])
        cot_i2 = 1.0 / np.tan(angles[:, i2])
        cot_i3 = 1.0 / np.tan(angles[:, i3])

        # Poids de l'arête (factore) et de la diagonale (factord)
        factore = -0.5 * e1_D_e2 / sin_i3
        factord = -0.5 * e1_D_e1 * (cot_i2 + cot_i3)

        # Accumulation dans les listes au format COO
        # W[i1, i2] += factore
        I_W.append(F[:, i1])
        J_W.append(F[:, i2])
        V_W.append(factore)

        # W[i2, i1] += factore
        I_W.append(F[:, i2])
        J_W.append(F[:, i1])
        V_W.append(factore)

        # W[i1, i1] += factord
        I_W.append(F[:, i1])
        J_W.append(F[:, i1])
        V_W.append(factord)

    I_W = np.concatenate(I_W)
    J_W = np.concatenate(J_W)
    V_W = np.concatenate(V_W)

    # scipy.sparse.coo_matrix somme automatiquement les doublons (i, j)
    W = sp.coo_matrix((V_W, (I_W, J_W)), shape=(n_vertices, n_vertices)).tocsr()


    return W.tocsr()

In [58]:
def compute_mass_matrix(V: np.ndarray, F: np.ndarray) -> sp.csr_matrix:
    n_vertices = V.shape[0]

    # 1. Extraction vectorisée en une seule passe
    v1, v2, v3 = V[F].transpose(1, 0, 2)

    # 2. Calcul des aires
    triangle_areas = 0.5 * np.linalg.norm(np.cross(v2 - v1, v3 - v1), axis=1)

    # 3. Distribution aux sommets
    vertex_masses = np.bincount(
        F.flatten(),
        weights=np.repeat(triangle_areas / 3.0, 3),
        minlength=n_vertices
    )

    # 4. Matrice creuse
    return sp.diags(vertex_masses, format='csr')

## 5. Direct Heat Diffusion Simulation (Spatial Domain)

Before computing the spectral decomposition of the FLBO, we can directly observe its physical behavior by simulating heat diffusion on the manifold. The heat equation is governed by:

$$\frac{\partial f}{\partial t} = -\Delta_{FLBO} f = -S_{FLBO}^{-1} W_{FLBO} f$$

To simulate this numerically over time without relying on the spectral domain, we use the unconditionally stable Backward Euler implicit scheme. Given an initial heat distribution $f_0$ (e.g., a Dirac delta at a specific vertex) and a time step $t$, we solve the following sparse linear system for $f_t$:

$$(S_{FLBO} + t W_{FLBO}) f_t = S_{FLBO} f_0$$

By varying the anisotropy parameter $\alpha$ and the orientation angle $\theta$, we can visualize how the Randers metric actively steers the heat flow along specific directions on the surface, breaking the symmetric diffusion seen in classical Riemannian geometry.

In [59]:
def solve_heat_diffusion(V, F, source, t, options, U = None):
    if U is None:
        U = compute_local_frames(V, F)
    D_F = compute_D_finsler(U, options)
    W = compute_stiffness_matrix(V, F, D_F)
    S = compute_mass_matrix(V, F)

    f0 = np.zeros(V.shape[0])
    f0[source] = 1.0

    return sla.spsolve(S - t * W, S @ f0)

In [60]:
ft = solve_heat_diffusion(V, F, centre, 1, options=Options(alpha=0, angle=0, tau=0.1))

In [61]:
plot_mesh(V, F, vertex_func=ft)

In [62]:
def plot_diffusions_comparison(V, F, source, t, options_list):
    """
    Affiche la diffusion de chaleur pour une liste d'Options côte à côte.
    """
    V = np.asarray(V)
    F = np.asarray(F)
    num_plots = len(options_list)

    # 1. Création des titres dynamiques basés sur les paramètres
    titles = [f"alpha={opt.alpha}<br>angle={opt.angle:.2f}, tau={opt.tau}" for opt in options_list]

    # 2. Initialisation de la figure avec des subplots 3D (type 'scene')
    fig = make_subplots(
        rows=1, cols=num_plots,
        specs=[[{'type': 'scene'}] * num_plots],
        subplot_titles=titles,
        horizontal_spacing=0.01
    )

    # 3. Pré-calcul des arêtes (inutile de le refaire pour chaque subplot)
    edges = np.vstack([F[:, [0, 1]], F[:, [1, 2]], F[:, [2, 0]]])
    unique_edges = np.unique(np.sort(edges, axis=1), axis=0)

    x_edges, y_edges, z_edges = np.empty(3 * len(unique_edges)), np.empty(3 * len(unique_edges)), np.empty(3 * len(unique_edges))
    x_edges[0::3], x_edges[1::3], x_edges[2::3] = V[unique_edges[:, 0], 0], V[unique_edges[:, 1], 0], np.nan
    y_edges[0::3], y_edges[1::3], y_edges[2::3] = V[unique_edges[:, 0], 1], V[unique_edges[:, 1], 1], np.nan
    z_edges[0::3], z_edges[1::3], z_edges[2::3] = V[unique_edges[:, 0], 2], V[unique_edges[:, 1], 2], np.nan

    # 4. Boucle sur les options pour générer et ajouter les traces
    for i, opt in enumerate(options_list):
        # Calcul de la diffusion
        ft = solve_heat_diffusion(V, F, source, t, options=opt)

        # Trace du maillage avec les couleurs
        mesh_trace = go.Mesh3d(
            x=V[:, 0], y=V[:, 1], z=V[:, 2],
            i=F[:, 0], j=F[:, 1], k=F[:, 2],
            intensity=ft,
            intensitymode='vertex',
            colorscale='Viridis',
            opacity=1.0,
            flatshading=True,
            name=f'Finsler {i+1}',
            showscale=(i == num_plots - 1)  # Affiche la barre de couleur uniquement sur le dernier graphe
        )

        # Trace des arêtes noires
        edge_trace = go.Scatter3d(
            x=x_edges, y=y_edges, z=z_edges,
            mode='lines',
            line=dict(color='black', width=2),
            hoverinfo='skip',
            showlegend=False
        )

        # Ajout des traces dans la bonne colonne (Plotly commence à 1)
        col_idx = i + 1
        fig.add_trace(mesh_trace, row=1, col=col_idx)
        fig.add_trace(edge_trace, row=1, col=col_idx)

    # 5. Ajustement des axes pour qu'ils aient les mêmes proportions
    scene_layout = dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        aspectmode='data'  # Garde les vraies proportions de la géométrie
    )

    # Applique le layout à toutes les scènes créées dynamiquement (scene, scene2, scene3...)
    fig.update_layout(
        **{f"scene{j+1 if j > 0 else ''}": scene_layout for j in range(num_plots)},
        margin=dict(l=0, r=0, b=0, t=50),
    )

    return fig

In [63]:
plot_diffusions_comparison(
    V, F, centre, 1,
    options_list=[
        Options(alpha=10, angle=0, tau=0.1),
        Options(alpha=10, angle=np.pi/4, tau=0.1),
    ]
)